# 🫀 Multimodal Heart Failure Readmission Prediction Pipeline
**Master Kaggle End-to-End Execution Notebook**

This notebook executes the entire pipeline sequentially:
1. **Setup & Clone Repository**
2. **Copy Teammates Parquet Cohort Splits**
3. **Generate Fast Mock Clinical Tables** (Labs & Vitals)
4. **Train Tabular Branch** (XGBoost Bootstrap Ensemble)
5. **Train ECG Branch** (1D ResNet-34 with MC-Dropout)
6. **Train CXR Branch** (DenseNet-121 with Transfer Learning)
7. **Train Gated Fusion Layer** (Masked Softmax MLP Gating)
8. **Run Comprehensive Evaluations** (DCA, Baselines, Confusion Matrices, Fairness)
9. **Generate & Display Inline Dashboard**

In [ ]:
# ── Cell 1: Environment Setup & Clone ──────────────────────────────────
import os, shutil

if not os.path.exists("HealthCare_Analytics"):
    !git clone https://github.com/KMohnishM/HealthCare_Analytics.git

%cd HealthCare_Analytics
!git pull

In [ ]:
# ── Cell 2: Copy Teammate Parquet Cohort Data ──────────────────────────
import os, glob, shutil

os.makedirs("data", exist_ok=True)
for input_path in glob.glob("/kaggle/input/**/*.parquet", recursive=True):
    filename = os.path.basename(input_path)
    dest_path = os.path.join("data", filename)
    shutil.copy(input_path, dest_path)
    print(f"Copied {filename} -> {dest_path}")

In [ ]:
# ── Cell 3: Generate Fast Mock Clinical Tables ──────────────────────────
!python scripts/generate_mock_clinical_tables.py

In [ ]:
# ── Cell 4: Train Tabular Branch (XGBoost Ensemble) ───────────────────
!python scripts/train_tabular.py

In [ ]:
# ── Cell 5: Train ECG Branch (1D ResNet-34) ───────────────────────────
!python scripts/train_ecg.py

In [ ]:
# ── Cell 6: Train CXR Branch (DenseNet-121) ───────────────────────────
!python scripts/train_cxr.py

In [ ]:
# ── Cell 7: Train Gated Fusion Model (MLP Gating Head) ─────────────────
!python scripts/train_fusion.py

In [ ]:
# ── Cell 8: Comprehensive Evaluation (DCA, Baselines, Confusion Matrix) ──
!python scripts/evaluate_all.py

In [ ]:
# ── Cell 9: Generate Interactive Dashboard Notebook ────────────────────
!python scripts/generate_notebook.py

In [ ]:
# ── Cell 10: Display All Visual Results Inline in Kaggle ───────────────
import os
from IPython.display import Image, display, HTML

figures = [
    ("confusion_matrix.png", "Side-by-Side Confusion Matrices (F1-Optimized Thresholds)"),
    ("decision_curve.png", "Clinical Decision Curve Analysis (DCA vs. LACE / HOSPITAL)"),
    ("missingness_sweep_heatmap.png", "Modality Missingness Sweep Heatmap"),
    ("fairness_subgroups.png", "Algorithmic Fairness Subgroup Analysis")
]

for filename, title in figures:
    filepath = os.path.join("outputs", "figures", filename)
    if os.path.exists(filepath):
        display(HTML(f"""<h3 style='color:#2c3e50; font-family:sans-serif; border-bottom: 2px solid #ecf0f1; padding-bottom: 5px;'>
                        {title} (<code>{filename}</code>)
                      </h3>"""))
        display(Image(filename=filepath, width=750))
    else:
        print(f"Warning: Figure {filename} not found at {filepath}")